# Convolutional Neural Networks

## Project: Write an Algorithm for Landmark Classification

### A simple app

In this notebook we build a very simple app that uses our exported model.

> <img src="static_images/icons/noun-info-2558213.png" alt="?" style="width:25px"/> Note how we are not importing anything from our source code (we do not use any module from the ``src`` directory). This is because the exported model, differently from the model weights, is a standalone serialization of our model and therefore it does not need anything else. You can ship that file to anybody, and as long as they can import ``torch``, they will be able to use your model. This is very important for releasing pytorch models to production.

In [1]:
from ipywidgets import VBox, Button, FileUpload, Output, Label
from PIL import Image
from IPython.display import display
import io
import numpy as np
import torchvision
import torchvision.transforms as T
import torch

# Load exported transfer learning model
learn_inf = torch.jit.load("checkpoints/transfer_exported.pt")
learn_inf.eval()

def get_uploaded_file_content(upload_widget):
    """
    Compatible helper for different ipywidgets FileUpload versions.
    Works with both dictionary-based and tuple-based upload values.
    """
    uploaded = upload_widget.value

    if uploaded is None or len(uploaded) == 0:
        return None

    # ipywidgets 8 usually returns a tuple of dictionaries
    if isinstance(uploaded, tuple):
        return uploaded[-1]["content"]

    # Some versions return a dictionary
    if isinstance(uploaded, dict):
        first_key = list(uploaded.keys())[-1]
        return uploaded[first_key]["content"]

    return None


def on_click_classify(change):

    # Load uploaded image
    file_content = get_uploaded_file_content(btn_upload)

    if file_content is None:
        out_pl.clear_output()
        with out_pl:
            print("Please upload an image first.")
        return

    fn = io.BytesIO(file_content)

    img = Image.open(fn).convert("RGB")
    img.load()

    # Clear previous output
    out_pl.clear_output()

    # Display uploaded image
    with out_pl:
        ratio = img.size[0] / img.size[1]
        c = img.copy()
        c.thumbnail((int(ratio * 200), 200))
        display(c)

    # Transform image to tensor
    timg = T.ToTensor()(img).unsqueeze(0)

    # Run inference
    with torch.no_grad():
        softmax = learn_inf(timg).data.cpu().numpy().squeeze()

    idxs = np.argsort(softmax)[::-1]

    # Show top 5 predictions
    for i in range(5):
        p = softmax[idxs[i]]
        landmark_name = learn_inf.class_names[idxs[i]]
        labels[i].value = f"{landmark_name} (prob: {p:.2f})"


# Upload widget
btn_upload = FileUpload(
    accept="image/*",
    multiple=False
)

btn_run = Button(description="Classify")
btn_run.on_click(on_click_classify)

labels = []
for _ in range(5):
    labels.append(Label())

out_pl = Output()
out_pl.clear_output()

wgs = [
    Label("Please upload a picture of a landmark"),
    btn_upload,
    btn_run,
    out_pl
]

wgs.extend(labels)

VBox(wgs)


The app successfully loads the exported TorchScript transfer learning model and uses it to classify an uploaded landmark image. The main strength of this approach is that the model is self-contained: it can be loaded with `torch.jit.load` without importing the original source code from the `src` directory. This makes the model easier to deploy and share.

The transfer learning model is also more effective than the CNN trained from scratch because it reuses visual features learned from a large ImageNet dataset. This helps the model identify visual patterns such as shapes, textures, edges, and architectural structures that are useful for landmark classification.

However, the model still has limitations. It can confuse landmarks that look visually similar, especially if they share architectural styles, colors, or natural surroundings. The model may also perform worse on images with unusual angles, poor lighting, occlusions, or landmarks that occupy only a small part of the image. Another limitation is that the model can only predict one of the 50 landmark classes included in the training dataset, so images of landmarks outside those classes may still receive a high-confidence but incorrect prediction.

## (optional) Taking it to the next level

You can run this notebook as a standalone app on your computer by following these steps:

1. Download this notebook in a directory on your machine
2. Download the model export (for example, ``checkpoints/transfer_exported.pt``) in a subdirectory called ``checkpoints`` within the directory where you save the app.ipynb notebook
3. Install voila if you don't have it already (``pip install voila``)
4. Run your app: ``voila app.ipynb --show_tracebacks=True``
5. Customize your notebook to make your app prettier and rerun voila

You can also deploy this app as a website using Binder: https://voila.readthedocs.io/en/stable/deploy.html#deployment-on-binder

In [2]:
from pathlib import Path

required_files = [
    "Project_Landmarks_Part1_CNNfromScratch__starter.ipynb",
    "Project_Landmarks_Part2_TransferLearning__starter.ipynb",
    "Project_Landmarks_Part3_App__starter.ipynb",
    "src/train.py",
    "src/model.py",
    "src/helpers.py",
    "src/predictor.py",
    "src/transfer.py",
    "src/optimization.py",
    "src/data.py",
    "checkpoints/original_exported.pt",
    "checkpoints/transfer_exported.pt",
]

for file in required_files:
    path = Path(file)
    print(f"{'✅' if path.exists() else '❌'} {file}")

✅ Project_Landmarks_Part1_CNNfromScratch__starter.ipynb
✅ Project_Landmarks_Part2_TransferLearning__starter.ipynb
✅ Project_Landmarks_Part3_App__starter.ipynb
✅ src/train.py
✅ src/model.py
✅ src/helpers.py
✅ src/predictor.py
✅ src/transfer.py
✅ src/optimization.py
✅ src/data.py
✅ checkpoints/original_exported.pt
✅ checkpoints/transfer_exported.pt


In [3]:
!pytest src/data.py src/model.py src/optimization.py src/train.py src/predictor.py src/transfer.py

============================= test session starts ==============================
platform linux -- Python 3.12.12, pytest-8.4.2, pluggy-1.6.0
rootdir: /workspace/code/project
configfile: pytest.ini
plugins: anyio-4.12.1
collected 18 items                                                             

src/data.py ....                                                         [ 22%]
src/model.py .                                                           [ 27%]
src/optimization.py .......                                              [ 66%]
src/train.py ....                                                        [ 88%]
src/predictor.py .                                                       [ 94%]
src/transfer.py .                                                        [100%]

=============================== warnings summary ===============================
src/data.py::test_data_loaders_keys
src/data.py::test_visualize_one_batch
src/model.py::test_model_construction
src/train.py::test_train_

In [ ]:
import tarfile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y-%m-%dT%Hh%Mm")
submission_name = f"submission_{timestamp}.tar.gz"

files_to_include = [
    "Project_Landmarks_Part1_CNNfromScratch__starter.ipynb",
    "Project_Landmarks_Part2_TransferLearning__starter.ipynb",
    "Project_Landmarks_Part3_App__starter.ipynb",
    "src/train.py",
    "src/model.py",
    "src/helpers.py",
    "src/predictor.py",
    "src/transfer.py",
    "src/optimization.py",
    "src/data.py",
    "checkpoints/original_exported.pt",
    "checkpoints/transfer_exported.pt",
]

with tarfile.open(submission_name, "w:gz") as tar:
    for file in files_to_include:
        path = Path(file)
        if path.exists():
            tar.add(path, arcname=file)
            print(f"Added: {file}")
        else:
            print(f"Missing, not added: {file}")

print(f"\nCreated submission archive: {submission_name}")